# Build Water GSA Dictionary

Builds `water_gsa_dictionary.csv` from DWR's **i03 Groundwater Sustainability Agencies** dataset
(California Open Data), plus a small set of hand-written SGMA-wide concept rows.

Each output row has:
- `all_names`: pipe-separated full name + aliases (the R pipeline splits on `|`)
- `entity_type`: `sgma_concept` | `gsa`
- `notes`: subbasin association(s) from the i03 dataset

## Data source
Live download from the DWR i03 CSV. There is no local snapshot — re-run the
notebook to pick up the latest entries from the SGMA Portal.

- DWR i03 GSA dataset (CSV): https://gis.data.cnra.ca.gov/api/download/v1/items/51688234e6104427894328acee4983e8/csv?layers=0
- DWR SGMA Portal: https://sgma.water.ca.gov/portal/
- California Open Data page: https://data.ca.gov/dataset/i03-groundwater-sustainability-agencies

## Cross-dict overlap
This dict is intentionally **not** deduped against sibling dictionaries at build time. Each dict aims to be self-contained — a comprehensive list of its category. Cross-dict alias collisions are handled at runtime in step4's `global_dict` construction and duplicate-`from` resolution loop.


In [ ]:
import os, re, csv, io, urllib.request
from collections import defaultdict, Counter

# === Paths ===
DICTS_DIR = "."                                  # this folder
OUT_CSV   = os.path.join(DICTS_DIR, "water_gsa_dictionary.csv")

I03_URL = ("https://gis.data.cnra.ca.gov/api/download/v1/items/"
           "51688234e6104427894328acee4983e8/csv?layers=0")


## Pull the i03 dataset (in-memory)


In [ ]:
print(f"Fetching i03 GSA CSV from {I03_URL}")
with urllib.request.urlopen(I03_URL) as r:
    csv_bytes = r.read()
print(f"  -> {len(csv_bytes):,} bytes")
i03_rows = list(csv.DictReader(io.StringIO(csv_bytes.decode("utf-8-sig"))))
print(f"  -> {len(i03_rows)} GSA records")


## SGMA-wide concept rows

These aren't in the i03 dataset; they're prepended so the entity ruler picks up generic SGMA terminology in the GSP corpus.


In [ ]:
sgma_concepts = [
    ("Groundwater Sustainability Agency|GSA",       "sgma_concept", "generic"),
    ("Groundwater Sustainability Agencies|GSAs",    "sgma_concept", "generic plural"),
    ("Groundwater Sustainability Plan|GSP",         "sgma_concept", "generic"),
    ("Groundwater Sustainability Plans|GSPs",       "sgma_concept", "generic plural"),
    ("Sustainable Groundwater Management Act|SGMA", "sgma_concept", "2014 statute"),
    ("DWR Bulletin 118|Bulletin 118",               "sgma_concept", "basin priority designations"),
]


## Collapse multi-area GSA filings into a single canonical row

The i03 dataset has one row per (GSA, subbasin). For e.g. Tehama County FCWCD that produces 5 rows; we want one GSA entry with all subbasins recorded in `notes` and all area-specific name variants as aliases.


In [ ]:
gsa_aliases = defaultdict(set)   # canonical -> set of name variants
gsa_basins  = defaultdict(set)   # canonical -> set of subbasin strings

def underlying(name):
    """'Westlands Water District GSA' -> 'Westlands Water District'"""
    return name[:-4].strip() if name.endswith(" GSA") else None

for r in i03_rows:
    full = r["GSA_Name"].strip()
    if not full:
        continue
    # Rows like "<Org> GSA - <area>" -> canonicalize on "<Org> GSA"
    m = re.match(r"^(.*?\bGSA)\s*-\s*(.+)$", full)
    canon = m.group(1).strip() if m else full
    gsa_aliases[canon].add(full)
    gsa_aliases[canon].add(canon)
    u = underlying(canon)
    if u:
        gsa_aliases[canon].add(u)
    b = (r["Basin_Subbasin_Name"] or "").strip().strip('"')
    if b:
        gsa_basins[canon].add(b)
print(f"Collapsed {len(i03_rows)} i03 rows into {len(gsa_aliases)} canonical GSAs")


## Build rows + write CSV


In [ ]:
out_rows = list(sgma_concepts)
for canon in sorted(gsa_aliases.keys()):
    aliases = sorted(gsa_aliases[canon])
    if canon in aliases:
        ordered = [canon] + [a for a in aliases if a != canon]
    else:
        ordered = sorted(aliases, key=len)
    notes = "; ".join(sorted(gsa_basins[canon])) if gsa_basins[canon] else ""
    out_rows.append(("|".join(ordered), "gsa", notes))

with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["all_names", "entity_type", "notes"])
    for row in out_rows:
        w.writerow(row)

by_type = Counter(r[1] for r in out_rows)
print(f"Wrote {len(out_rows)} rows -> {OUT_CSV}")
print(f"  by entity_type: {dict(by_type)}")
